In [ ]:
%cd ../../..

import os
import numpy as np
import random
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from tqdm import tqdm
import matplotlib.pyplot as plt

from evaluation import *

for i in range(torch.cuda.device_count()):
    print(f"Device {i}: {torch.cuda.get_device_name(i)}")

In [ ]:
data_path = "/scratch/VM/radio-foundation/preprocessed/LUNA16"
embed_path = "/scratch/VM/radio-foundation/embeddings/DINO3D/LUNA16/chp100k"

device_idx = 1
device = torch.device(f"cuda:{device_idx}")
print(f"Using Device {device_idx}: {torch.cuda.get_device_name(device_idx)}")

In [ ]:
NUM_EPOCHS = 30
BATCH_SIZE = 16
NUM_WORKERS = 6
LR = 0.0001
WEIGHT_DECAY = 0.01

THRESHOLDS = [0.05, 0.1, 0.2]

set_seeds(4)

In [ ]:
labels_df = pd.read_csv(os.path.join(data_path, "targets.csv"))
series_uids = labels_df["seriesuid"].unique().tolist()
random.shuffle(series_uids)
train_split = 0.8
num_train = int(train_split * len(series_uids))
train_series_uids = series_uids[:num_train]
valid_series_uids = series_uids[num_train:]

train_df = labels_df[labels_df["seriesuid"].isin(train_series_uids)].reset_index(drop=True)
valid_df = labels_df[labels_df["seriesuid"].isin(valid_series_uids)].reset_index(drop=True)

print(f"Train samples: {len(train_df)} - Valid samples: {len(valid_df)}")

In [ ]:
class EmbeddingDatasetLocation(Dataset):
    def __init__(self, labels_df, embed_path, add_noise=False, sigma=0.1, p=0.0):
        self.labels_df = labels_df
        self.embed_path = embed_path

        self.add_noise = add_noise
        self.sigma = sigma
        self.p = p

    def add_embedding_noise(self, embeddings):
        if torch.rand(1).item() < self.p:
            noise = torch.randn_like(embeddings) * self.sigma
            embeddings = embeddings + noise
        return embeddings

    def __len__(self):
        return len(self.labels_df)

    def __getitem__(self, idx):
        row = self.labels_df.iloc[idx]
        name = row["name"]
        rel_z = row["rel_z"]
        rel_y = row["rel_y"]
        rel_x = row["rel_x"]

        features = torch.load(os.path.join(self.embed_path, name.replace(".npy", ".pth")))["patch"]
        
        if self.add_noise:
            features = self.add_embedding_noise(features)

        return features, (rel_z, rel_y, rel_x)


In [ ]:
class PositionalAttentionRegressor3D(nn.Module):
    def __init__(self, embed_dim, n_heads=4, hidden_dim=128):
        super().__init__()
        self.d_model = embed_dim
        self.pos_linear = nn.Linear(3, embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=n_heads, batch_first=True)
        self.spatial_attn = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x):
        B, Nz, Ny, Nx, D = x.shape
        N = Nz * Ny * Nx
        x = x.view(B, N, D)

        grid_z, grid_y, grid_x = torch.meshgrid(
            torch.linspace(0, 1, Nz, device=x.device),
            torch.linspace(0, 1, Ny, device=x.device),
            torch.linspace(0, 1, Nx, device=x.device),
            indexing="ij"
        )
        pos = torch.stack([grid_z, grid_y, grid_x], dim=-1).view(1, N, 3).expand(B, N, 3)

        x = x + self.pos_linear(pos)

        out, _ = self.attn(x, x, x)
        attn_weights = torch.softmax(self.spatial_attn(out), dim=1)  # (B, N, 1)

        coords = (attn_weights * pos).sum(dim=1)
        return coords



In [ ]:
train_ds = EmbeddingDatasetLocation(train_df, embed_path, add_noise=True, sigma=0.1, p=1.0)
valid_ds = EmbeddingDatasetLocation(valid_df, embed_path, add_noise=False)

def collate_fn(batch):
    feats = [torch.as_tensor(b[0]) for b in batch]
    targets = [torch.tensor(b[1], dtype=torch.float32) for b in batch]
    feats = torch.stack([f.float() for f in feats], dim=0)
    targets = torch.stack(targets, dim=0)
    return feats, targets

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True, 
    collate_fn=collate_fn
)
valid_loader = DataLoader(
    valid_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    collate_fn=collate_fn
)

In [ ]:
sample_feat, _ = train_ds[0]
embed_dim = sample_feat.shape[-1]

model = PositionalAttentionRegressor3D(embed_dim=embed_dim, n_heads=4, hidden_dim=256)
model = model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

mse_loss = nn.MSELoss()

def euclidean_errors(preds, targets):
    dists = torch.sqrt(((preds - targets) ** 2).sum(dim=1)).detach().cpu().numpy()
    return dists

def accuracy_within_threshold(preds, targets, thresh):
    dists = euclidean_errors(preds, targets)
    return float((dists <= thresh).mean())

best_val_loss = float("inf")
for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    train_losses = []
    pbar = tqdm(train_loader, desc=f"Epoch {epoch} train", leave=False)
    for feats, targets in pbar:
        feats = feats.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad()
        
        preds = model(feats)
        loss = mse_loss(preds, targets)

        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())
        pbar.set_postfix(train_loss=np.mean(train_losses))

    avg_train_loss = float(np.mean(train_losses))

    model.eval()
    val_losses = []
    all_preds = []
    all_targets = []
    with torch.no_grad():
        vbar = tqdm(valid_loader, desc=f"Epoch {epoch} val", leave=False)
        for feats, targets in vbar:
            feats = feats.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            preds = model(feats)
            loss = mse_loss(preds, targets)
            val_losses.append(loss.item())
            all_preds.append(preds.detach().cpu())
            all_targets.append(targets.detach().cpu())

    avg_val_loss = float(np.mean(val_losses))
    all_preds = torch.cat(all_preds, dim=0)
    all_targets = torch.cat(all_targets, dim=0)
    dists = euclidean_errors(all_preds, all_targets)
    mean_dist = float(dists.mean())
    median_dist = float(np.median(dists))
    accs = {thr: float((dists <= thr).mean()) for thr in THRESHOLDS}

    print(f"Epoch {epoch:02d} | train_loss: {avg_train_loss:.6f} | val_loss: {avg_val_loss:.6f} | "
          f"mean_dist: {mean_dist:.5f} | median_dist: {median_dist:.5f} | accs: {accs}")

In [ ]:
idx = 0
row = valid_df.iloc[idx]

vol_path = os.path.join(data_path, row["name"])
new_volume = np.load(vol_path)
rel_z, rel_y, rel_x = row["rel_z"], row["rel_y"], row["rel_x"]

embed_file = os.path.join(embed_path, row["name"].replace(".npy", ".pth"))
features = torch.load(embed_file)["patch"].unsqueeze(0).to(device)

model.eval()
with torch.no_grad():
    pred_coords = model(features).cpu().numpy()[0]

print(f"True coords: z={rel_z:.4f}, y={rel_y:.4f}, x={rel_x:.4f}")
print(f"Pred coords: z={pred_coords[0]:.4f}, y={pred_coords[1]:.4f}, x={pred_coords[2]:.4f}")

z_t, y_t, x_t = int(rel_z * new_volume.shape[0]), int(rel_y * new_volume.shape[1]), int(rel_x * new_volume.shape[2])
z_p, y_p, x_p = int(pred_coords[0] * new_volume.shape[0]), int(pred_coords[1] * new_volume.shape[1]), int(pred_coords[2] * new_volume.shape[2])

vox_dist = np.sqrt((z_t - z_p)**2 + (y_t - y_p)**2 + (x_t - x_p)**2)
print(f"Voxel distance between prediction and target: {vox_dist:.2f} voxels")

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
v_min, v_max = -1000, 400

axes[0].imshow(new_volume[z_t, :, :], cmap='gray', origin='lower', vmin=v_min, vmax=v_max)
axes[0].plot(x_t, y_t, 'r+', markersize=14, alpha=0.7)
axes[0].plot(x_p, y_p, 'gx', markersize=12, alpha=0.7)
axes[0].set_title(f"Axial (Z={z_t})")

axes[1].imshow(new_volume[:, y_t, :], cmap='gray', origin='lower', vmin=v_min, vmax=v_max)
axes[1].plot(x_t, z_t, 'r+', markersize=14, alpha=0.7)
axes[1].plot(x_p, z_p, 'gx', markersize=12, alpha=0.7)
axes[1].set_title(f"Coronal (Y={y_t})")

axes[2].imshow(new_volume[:, :, x_t], cmap='gray', origin='lower', vmin=v_min, vmax=v_max)
axes[2].plot(y_t, z_t, 'r+', markersize=14, alpha=0.7)
axes[2].plot(y_p, z_p, 'gx', markersize=12, alpha=0.7)
axes[2].set_title(f"Sagittal (X={x_t})")

fig.suptitle(f"{row['seriesuid']}", fontsize=16)

plt.show()